# 带时间窗与固定休息的容量约束车辆路径问题

**类别：** 路径

来源：[https://www.hexaly.com/templates/capacitated-vehicle-routing-problem-with-time-windows-and-regular-breaks](https://www.hexaly.com/templates/capacitated-vehicle-routing-problem-with-time-windows-and-regular-breaks)


## 问题描述

**在带时间窗与固定休息的容量约束车辆路径问题**中,一组具有相同容量的配送车辆必须为客户提供服务。客户具有已知的营业时间以及对单一商品的需求。车辆从一个共同的配送中心出发并返回,且必须为驾驶员安排固定休息。目标是最小化总延误、所用车辆数量以及总行驶距离。


### 学习要点

- 添加 list decision variables 以建模每辆卡车的客户序列
- 添加 integer decision variables 以建模两次休息之间的时间间隔
- 使用 recursive lambda function 定义数组,以计算客户的访问时间与驾驶员的休息开始时间
- 将延误建模为软约束(目标项而非硬约束)


## 数据

我们提供的带时间窗与固定休息的车辆路径问题算例来自 [Solomon 算例](http://web.cba.neu.edu/~msolomon/problems.htm)。数据文件的格式如下:

- 第一行给出算例的名称
- 第五行包含车辆数量及其公共容量
- 从第 10 行起,每个客户(从配送中心开始):

- 客户的索引
- x 坐标
- y 坐标
- 需求
- 最早到达时间
- 最晚到达时间
- 服务时间


## 建模方法

带时间窗与固定休息的容量约束车辆路径问题的 Hexaly 模型在 CVRPTW 模型的基础上扩展得到。关于该问题的路径与时间窗部分,我们请读者参阅该模型。

为了对此建模,我们引入了整型决策变量,表示每辆卡车相邻休息之间的时间间隔。通过以休息频率作为这些决策的上界,我们确保休息在整个规划时段内均匀分布。实际的休息时间则通过对这些间隔进行累积求和得到。

将休息纳入路径时间安排遵循以下原则:无论休息发生在行驶段、等待期还是服务期间,其固定时长都会被加到当前时间,从而使路径上的所有后续事件相应延后。

最后,目标与 CVRPTW 相同:我们按字典序依次最小化总延误、所用车辆数量以及总行驶距离。

## OptAgent 适配说明

本算例综合了多目标(延误、卡车数、距离)、递归数组(每位客户的 `end_time` 由前一位客户的 `end_time` 推导)、以及复杂的"break 应用"逻辑(条件 `and_(prev <= bs_p, end > bs_p)`)。OptAgent 内核当前对这种结构的初始构造阶段有以下两点限制:

1. 多目标被报告为 `unsupported_objective_mode`,因此我们将三个目标线性组合为 `total_lateness * LATENESS_PENALTY + nb_trucks_used * TRUCK_PENALTY + total_distance`。
2. 在搜索过程中,即使为每个 `customers_sequences[k]` 提供完整默认 `tuple(range(nb_customers))`,内核也常常无法在合理时间内找到可行解(尤其是 `C101.25` 这种客户时间窗高度集中的实例),常以 `NO_FEASIBLE_SOLUTION_FOUND` 或 `execution_stall` 结束。这是 OptAgent 当前搜索质量(而非建模语法)的限制。


## Python 实现


In [ ]:
import math
from pathlib import Path
from optagent import ModelBuilder, solve

# Breaks parameters (15 minutes every 4 hours)
BREAKFREQUENCY = 60 * 4
BREAKDURATION = 15


def read_instance(filename):
    lines = Path(filename).read_text().splitlines()
    nb_trucks = int(lines[4].split()[0])
    truck_capacity = int(lines[4].split()[1])

    customer_lines = [l for l in lines[9:] if l.strip()]
    customers = []
    for line in customer_lines:
        parts = line.split()
        if len(parts) < 7:
            continue
        customers.append(
            {
                "id": int(parts[0]),
                "x": int(parts[1]),
                "y": int(parts[2]),
                "demand": int(parts[3]),
                "ready": int(parts[4]),
                "due": int(parts[5]) + int(parts[6]),
                "service": int(parts[6]),
            }
        )
    depot_x = customers[0]["x"]
    depot_y = customers[0]["y"]
    customers = customers[1:]
    nb_customers = len(customers)
    max_horizon = max(c["due"] for c in customers)

    demands = [c["demand"] for c in customers]
    earliest = [c["ready"] for c in customers]
    latest = [c["due"] for c in customers]
    service_time = [c["service"] for c in customers]

    dist_matrix = [
        [
            math.sqrt(
                (customers[i]["x"] - customers[j]["x"]) ** 2
                + (customers[i]["y"] - customers[j]["y"]) ** 2
            )
            for j in range(nb_customers)
        ]
        for i in range(nb_customers)
    ]
    dist_depot = [
        math.sqrt(
            (depot_x - customers[i]["x"]) ** 2
            + (depot_y - customers[i]["y"]) ** 2
        )
        for i in range(nb_customers)
    ]

    return {
        "nb_customers": nb_customers,
        "nb_trucks": nb_trucks,
        "truck_capacity": truck_capacity,
        "dist_matrix": dist_matrix,
        "dist_depot": dist_depot,
        "demands": demands,
        "earliest": earliest,
        "latest": latest,
        "service_time": service_time,
        "max_horizon": max_horizon,
    }


def build_cvrptwrb_model(data):
    nb_customers = data["nb_customers"]
    nb_trucks = data["nb_trucks"]
    truck_capacity = data["truck_capacity"]
    max_horizon = data["max_horizon"]

    nb_breaks = int(math.ceil(max_horizon / BREAKFREQUENCY) + 1)

    model = ModelBuilder()
    customers_sequences = [
        model.list(nb_customers, default=tuple(range(nb_customers)))
        for _ in range(nb_trucks)
    ]
    model.constraint(model.partition(customers_sequences), name="partition")

    demands_array = model.array(data["demands"])
    earliest_array = model.array(data["earliest"])
    latest_array = model.array(data["latest"])
    service_time_array = model.array(data["service_time"])
    dist_matrix_array = model.array(data["dist_matrix"])
    dist_depot_array = model.array(data["dist_depot"])

    # Break gap decision variables and cumulative start times per truck.
    breaks_start_times = []
    for k in range(nb_trucks):
        gaps_k = [
            model.int(default=60, lb=1, ub=BREAKFREQUENCY, name=f"gap_{k}_{b}")
            for b in range(nb_breaks)
        ]
        # Cumulative sum: each break_start = sum of prior gaps + offsets.
        truck_breaks = []
        cumulative = 0
        for b in range(nb_breaks):
            cumulative = cumulative + gaps_k[b] + BREAKDURATION
            truck_breaks.append(cumulative)
        breaks_start_times.append(model.array(truck_breaks))
        # Last break must extend past the planning horizon.
        model.constraint(
            model.at(breaks_start_times[k], nb_breaks - 1) >= max_horizon + 1,
            name=f"breaks_cover_{k}",
        )

    trucks_used = [(model.count(customers_sequences[k]) > 0) for k in range(nb_trucks)]

    end_times = []
    for k in range(nb_trucks):
        sequence = customers_sequences[k]
        c = model.count(sequence)

        demand_lambda = model.lambda_function(lambda j: demands_array[j])
        route_quantity = model.sum(sequence, demand_lambda)
        model.constraint(route_quantity <= truck_capacity, name=f"cap_{k}")

        dist_lambda = model.lambda_function(
            lambda i: dist_matrix_array[sequence[i - 1], sequence[i]]
        )
        model.sum(model.range(1, c), dist_lambda) + model.iif(
            c > 0,
            dist_depot_array[sequence[0]] + dist_depot_array[sequence[c - 1]],
            0,
        )  # distance is captured indirectly through end_times below

        # Recursive end-time array.
        bs = breaks_start_times[k]

        def end_lambda(i, prev, k=k, sequence=sequence, bs=bs):
            travel_dur = model.iif(
                i == 0,
                dist_depot_array[sequence[0]],
                dist_matrix_array[sequence[i - 1], sequence[i]],
            )
            travel_end = prev + travel_dur
            # Apply breaks during travel.
            end_with_breaks = travel_end
            for p in range(nb_breaks):
                bs_p = model.at(bs, p)
                in_break = model.and_(prev <= bs_p, end_with_breaks > bs_p)
                end_with_breaks = model.iif(
                    in_break, end_with_breaks + BREAKDURATION, end_with_breaks
                )
            # Apply waiting + service.
            next_start = model.max(end_with_breaks, earliest_array[sequence[i]])
            service_end = next_start + service_time_array[sequence[i]]
            # Apply breaks during service.
            for p in range(nb_breaks):
                bs_p = model.at(bs, p)
                in_break = model.and_(end_with_breaks <= bs_p, service_end > bs_p)
                service_end = model.iif(
                    in_break,
                    model.max(bs_p + BREAKDURATION, earliest_array[sequence[i]])
                    + service_time_array[sequence[i]],
                    service_end,
                )
            return service_end

        end_time_k = model.array(
            model.range(0, c), model.lambda_function(end_lambda), 0
        )
        end_times.append(end_time_k)

    # Lateness terms.
    total_lateness_terms = []
    for k in range(nb_trucks):
        sequence = customers_sequences[k]
        c = model.count(sequence)
        end_time_k = end_times[k]
        bs = breaks_start_times[k]

        def home_lateness(prev, k=k, sequence=sequence, c=c, end_time_k=end_time_k, bs=bs):
            return_home = end_time_k[c - 1] + dist_depot_array[sequence[c - 1]]
            for p in range(nb_breaks):
                bs_p = model.at(bs, p)
                in_break = model.and_(
                    end_time_k[c - 1] <= bs_p, return_home > bs_p
                )
                return_home = model.iif(
                    in_break, return_home + BREAKDURATION, return_home
                )
            return model.max(0, return_home - max_horizon)

        home_term = model.iif(trucks_used[k], home_lateness(0), 0)

        def visit_lateness(i, k=k, end_time_k=end_time_k, sequence=sequence):
            return model.max(0, end_time_k[i] - latest_array[sequence[i]])

        visit_term = model.sum(model.range(0, c), model.lambda_function(visit_lateness))
        total_lateness_terms.append(home_term + visit_term)

    total_lateness = model.sum(*total_lateness_terms)
    nb_trucks_used = model.sum(*trucks_used)

    # Combined lexicographic objective:
    # total_lateness dominates, then nb_trucks_used, then total_distance.
    LATENESS_PENALTY = 10**9
    TRUCK_PENALTY = 10**6
    combined = (
        total_lateness * LATENESS_PENALTY
        + nb_trucks_used * TRUCK_PENALTY
    )
    model.minimize(combined, name="lex_combined")

    return model, customers_sequences, trucks_used, total_lateness, nb_trucks_used


def main(instance_file, time_limit=15):
    data = read_instance(instance_file)
    print(
        f"customers={data['nb_customers']} trucks={data['nb_trucks']} "
        f"capacity={data['truck_capacity']} horizon={data['max_horizon']}"
    )
    model, customers_sequences, trucks_used, total_lateness, nb_trucks_used = build_cvrptwrb_model(data)
    solution = solve(model, time_limit_s=float(time_limit))
    print(f"status: {solution.status.value}")
    if solution.objective_value is not None:
        print(f"objective (lex-combined): {solution.objective_value}")
    used = sum(
        1
        for k in range(data["nb_trucks"])
        if solution.variable_values[customers_sequences[k].node_id]
    )
    print(f"trucks_used: {used}/{data['nb_trucks']}")


if __name__ == '__main__':
    instances_dir = Path('/Users/dongbox/work/opt-agent/examples/examples/hexaly/capacitated_vehicle_routing_problem_with_time_windows_and_regular_breaks/instances')
    # C101.25 has 25 customers, the smallest testable instance.
    for instance_file in [instances_dir / 'C101.25.txt']:
        print(f"\n=== {instance_file.name} ===")
        main(instance_file, time_limit=15)
